In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [2]:
df = pd.read_csv("DateFruit_Dataset.csv")

In [3]:
df.head() # 34 features, 1 categories

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [4]:
df.isnull().sum()

AREA             0
PERIMETER        0
MAJOR_AXIS       0
MINOR_AXIS       0
ECCENTRICITY     0
EQDIASQ          0
SOLIDITY         0
CONVEX_AREA      0
EXTENT           0
ASPECT_RATIO     0
ROUNDNESS        0
COMPACTNESS      0
SHAPEFACTOR_1    0
SHAPEFACTOR_2    0
SHAPEFACTOR_3    0
SHAPEFACTOR_4    0
MeanRR           0
MeanRG           0
MeanRB           0
StdDevRR         0
StdDevRG         0
StdDevRB         0
SkewRR           0
SkewRG           0
SkewRB           0
KurtosisRR       0
KurtosisRG       0
KurtosisRB       0
EntropyRR        0
EntropyRG        0
EntropyRB        0
ALLdaub4RR       0
ALLdaub4RG       0
ALLdaub4RB       0
Class            0
dtype: int64

In [5]:
X = df.drop("Class",axis=1)
y= df["Class"]
            

In [6]:
df["Class"].unique()

<StringArray>
['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY']
Length: 7, dtype: str

In [7]:
## Label Encoding:

In [8]:
le = LabelEncoder()
y=le.fit_transform(y)

In [9]:
## Train Test Split :

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [11]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [12]:
## DL

In [13]:
# Create Tensor

In [14]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)


In [15]:
#2. Create TensorDataset and DataLoader

In [16]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [17]:
### Creating DataLoader for the Tensor Dataset:

In [18]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [ ]:
# build our model

In [19]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(
            #1st Hidden Layer
            nn.Linear(X.shape[1],64),
            nn.ReLU(),
            # 2nd Hidden Layer
            nn.Linear(64,64),
            nn.ReLU(),
            nn.Linear(64,7)
        )

    def forward(self,x):
        return self.model(x)

In [20]:
model = ANN()

#loss and Optim
criteria=nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [21]:
## Training our Model

In [23]:
epochs = 100
for epoch in range (epochs):
    model.train()

    running_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()
        
        outputs = model(xb)
        loss = criteria(outputs, yb)
        loss.backward()
        optimizer.step() # params update

        running_loss += loss.item()  #.item() for convert tensor value to float value

    train_loss = running_loss/len(train_loader)

    print(f"epoch={epoch}/{epochs}, loss = {train_loss}")

epoch=0/100, loss = 1.6875521203745967
epoch=1/100, loss = 1.0837967447612598
epoch=2/100, loss = 0.7531796227330747
epoch=3/100, loss = 0.5619000066881594
epoch=4/100, loss = 0.4565533891968105
epoch=5/100, loss = 0.39594278646552045
epoch=6/100, loss = 0.3563681581745977
epoch=7/100, loss = 0.31970646847849304
epoch=8/100, loss = 0.2944241930609164
epoch=9/100, loss = 0.2657980394104253
epoch=10/100, loss = 0.2524766293556794
epoch=11/100, loss = 0.2381548058727513
epoch=12/100, loss = 0.22981860715409982
epoch=13/100, loss = 0.21279855461224265
epoch=14/100, loss = 0.2024536424356958
epoch=15/100, loss = 0.19336522921271945
epoch=16/100, loss = 0.183469632073589
epoch=17/100, loss = 0.18063872372326645
epoch=18/100, loss = 0.16940588536469833
epoch=19/100, loss = 0.16078808288211408
epoch=20/100, loss = 0.16615561104339102
epoch=21/100, loss = 0.15704804035308567
epoch=22/100, loss = 0.15311958608419998
epoch=23/100, loss = 0.15632321183448253
epoch=24/100, loss = 0.1465006830251735